# 1. 上下文增强

本节目标：在同一数据集上，先跑“纯向量检索 baseline”，明确失败形态，再依次引入 Sentence Window / Small-to-Big / AutoMerging 做对照。

## 本章要解决什么

在基础 RAG 中，检索器返回的片段往往"看起来相关，但读起来不够用"——命中了关键句，却丢失了前后的支撑信息。这不是检索没找对，而是切块粒度导致上下文被截断了。

举个例子：用户问"BN 和 LN 的区别"，检索器可能命中一句"LN 对单样本的所有维度做归一化"，但紧接着的对比说明（"BN 则是对一批样本的同一维度做归一化"）落在了另一个 chunk 里，最终答案就只有一半。

上下文增强要做的事情很简单：**在不改变检索逻辑的前提下，把命中片段周围的关键信息补回来**。本节会用同一份数据和同一批问题，依次尝试三种补回策略（Sentence Window / Small-to-Big / AutoMerging），并与 baseline 做可量化对比。

## 统一实验设置（一次定义，后面复用）

- 数据：`./data/face.pdf`
- 模型：`gpt-4o-mini`
- 比较目标：`baseline_df` vs `sentence_window_df` / `small_to_big_df` / `auto_merging_df`
- 评估：先做可读性强的规则评估，再保留后续可切换到 LLM 评估的接口

In [ ]:
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import Chroma

load_dotenv()

MODEL_NAME = "gpt-4o-mini"
PDF_PATH = Path("notebook/C7 高级 RAG 技巧/6. 增强阶段/data/face.pdf")

llm = ChatOpenAI(model=MODEL_NAME, temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

def load_chunks(chunk_size=256, chunk_overlap=40):
    docs = PyPDFLoader(str(PDF_PATH)).load()
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )
    return splitter.split_documents(docs)

def build_retriever(chunk_size=256, chunk_overlap=40, k=4):
    chunks = load_chunks(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    vs = Chroma.from_documents(chunks, embedding=embeddings)
    return vs.as_retriever(search_kwargs={"k": k})

qna_dict = {
    "介绍下SVM算法": "一种强调最大间隔的监督学习分类方法。",
    "BN 和 LN 区别": "BN 按 batch 维归一化，LN 按单样本特征维归一化。",
    "介绍 transformer 算法": "典型 encoder-decoder 架构，核心是自注意力机制。",
}

# 简化评估：按关键词命中判断，生产环境建议使用 LLM 评估或人工标注
def simple_eval(llm_answer: str, expected_answer: str) -> str:
    tokens = [t for t in expected_answer.replace("，", " ").replace("。", " ").split() if len(t) > 1]
    if not tokens:
        return "未评估"
    hit = sum(1 for t in tokens if t in llm_answer)
    return "意思一致" if hit >= max(1, len(tokens) // 2) else "意思不同"

## Baseline：纯向量检索先跑一遍

先不做任何上下文增强，只用基础向量检索回答 `qna_dict`。

失败观察重点：
- 命中内容是否相关；
- 最终回答是否“相关但不完整”。

In [ ]:
baseline_retriever = build_retriever(chunk_size=128, chunk_overlap=16, k=4)

baseline_rows = []
for question, expected in qna_dict.items():
    docs = baseline_retriever.invoke(question)
    context = "\n\n".join(d.page_content for d in docs)
    prompt = f"""
仅根据上下文回答问题。

问题：{question}
上下文：
{context}
"""
    answer = llm.invoke(prompt).content
    baseline_rows.append(
        {
            "question": question,
            "llm_answer": answer,
            "expected_answer": expected,
            "rag_eval_results": simple_eval(answer, expected),
        }
    )

baseline_df = pd.DataFrame(baseline_rows)
baseline_df

### Baseline 失败分析

观察上面的输出，你会发现 baseline 的回答方向大致正确——检索器确实命中了相关内容。但由于 chunk 只有 128 字符，很多关键的前后文被截断了：比如一个概念的定义和它的对比说明分别落在不同的 chunk 里，最终送给 LLM 的上下文是碎片化的。

这正是上下文增强要修复的典型失败模式：**不是"没找到"，而是"找到了但不完整"**。接下来我们用三种方法尝试把丢失的上下文补回来。

## 如何阅读本节（8 步模板）

后续每个方法都按同一结构展开：
1. 问题（失败场景）
2. 思路（为什么有效）
3. 流程（索引时/检索时）
4. 核心代码
5. 可运行示例
6. 结果对比（含 baseline）
7. 适用场景
8. 局限与边界

## Sentence Window

### 失败场景
命中句子本身相关，但答案依赖该句前后支撑句。

### 思路
把文档按句子切分并保存邻接关系；检索命中句后，在生成前恢复左右窗口。

### 索引时 vs 检索时
- 索引时：句子切分 + `id -> 邻居` 映射
- 检索时：先命中句子，再拼接左右窗口送给 LLM

### 适用边界
- 适合：局部上下文依赖强的说明文档
- 不适合：证据缺失、需要多步推理的问题

In [ ]:
# 核心逻辑：句子切分 -> 邻居映射 -> 命中后恢复窗口
import re

base_docs = PyPDFLoader(str(PDF_PATH)).load()
full_text = "\n".join(d.page_content for d in base_docs)
sentences = [s.strip() for s in re.split(r"(?<=[。！？!?])", full_text) if s.strip()]

sentence_map = {i: s for i, s in enumerate(sentences)}
neighbor_map = {
    i: [j for j in (i - 1, i, i + 1) if 0 <= j < len(sentences)]
    for i in sentence_map
}

sentence_vs = Chroma.from_texts(list(sentence_map.values()), embedding=embeddings)
sentence_retriever = sentence_vs.as_retriever(search_kwargs={"k": 2})

def sentence_window_answer(question: str) -> str:
    hits = sentence_retriever.invoke(question)
    hit_texts = [h.page_content for h in hits]
    hit_ids = [idx for idx, txt in sentence_map.items() if txt in hit_texts]
    window_ids = sorted({nid for hid in hit_ids for nid in neighbor_map.get(hid, [hid])})
    window_context = "\n".join(sentence_map[i] for i in window_ids)
    prompt = f"问题：{question}\n上下文：\n{window_context}\n请简洁作答。"
    return llm.invoke(prompt).content

rows = []
for question, expected in qna_dict.items():
    answer = sentence_window_answer(question)
    rows.append(
        {
            "question": question,
            "llm_answer": answer,
            "expected_answer": expected,
            "rag_eval_results": simple_eval(answer, expected),
        }
    )

sentence_window_df = pd.DataFrame(rows)
compare_q = list(qna_dict.keys())[0]
print("对比问题:", compare_q)
print("baseline:", baseline_df.loc[baseline_df['question'] == compare_q, 'llm_answer'].iloc[0][:120])
print("sentence_window:", sentence_window_df.loc[sentence_window_df['question'] == compare_q, 'llm_answer'].iloc[0][:120])

sentence_window_df

### Sentence Window 结果分析

对比 baseline 和 Sentence Window 的输出可以看到：当命中句的前后支撑句被恢复后，LLM 获得了更完整的论述链条，回答的覆盖度有明显提升。

Sentence Window 的核心价值在于**成本极低**——只需维护一个邻居映射，不改变检索逻辑。对于连续叙述型文档（说明文、技术报告），它是优先尝试的方案。

但它有一个天然限制：窗口是固定的（左右各 N 句），如果证据不在邻域而是分散在不同的段落层级——比如章节开头的总结和后续段落的细节——光扩大窗口并不能根本解决问题。这引出了下一个方法：Small-to-Big。

## Small-to-Big

### 失败场景
小块召回准但信息碎；大块信息全但相似度容易偏移。

### 思路
先按小块检索定位，再回填对应父块（段落级）做生成。

### 流程
1. 建 child chunks 用于召回
2. 建 `child_id -> parent_id` 映射
3. 命中 child 后回填 parent 内容

### 适用边界
- 适合：文档有稳定段落结构
- 不适合：文档结构极不规则、父子映射成本过高

In [ ]:
# 核心逻辑：child 检索 + parent 回填
paragraphs = [p.strip() for p in full_text.split("\n\n") if p.strip()]
parent_texts = paragraphs if paragraphs else [full_text]

child_texts = []
child_to_parent = {}
for p_idx, p in enumerate(parent_texts):
    parts = [p[i:i+180] for i in range(0, len(p), 150)]
    for part in parts:
        c_idx = len(child_texts)
        child_texts.append(part)
        child_to_parent[c_idx] = p_idx

child_vs = Chroma.from_texts(child_texts, embedding=embeddings)
child_retriever = child_vs.as_retriever(search_kwargs={"k": 4})

def small_to_big_answer(question: str) -> str:
    hits = child_retriever.invoke(question)
    hit_ids = [idx for idx, txt in enumerate(child_texts) if txt in {h.page_content for h in hits}]
    parent_ids = sorted({child_to_parent[i] for i in hit_ids})
    context = "\n\n".join(parent_texts[i] for i in parent_ids[:3])
    prompt = f"问题：{question}\n上下文：\n{context}\n请简洁作答。"
    return llm.invoke(prompt).content

rows = []
for question, expected in qna_dict.items():
    answer = small_to_big_answer(question)
    rows.append(
        {
            "question": question,
            "llm_answer": answer,
            "expected_answer": expected,
            "rag_eval_results": simple_eval(answer, expected),
        }
    )

small_to_big_df = pd.DataFrame(rows)
small_to_big_df

### Small-to-Big 结果分析

Small-to-Big 用小块做精确召回定位，再回填父块做生成——兼顾了"检索精度"和"上下文完整性"。从结果可以看到，当文档有清晰的段落结构时，父块回填能有效补全 Sentence Window 无法覆盖的跨段落信息。

不过 Small-to-Big 有一个隐含假设：每个子块只属于一个父块，且父块本身就是一个完整的语义单元。但实际文档中经常出现这样的情况：多个子块分散命中了同一个父块，单独看每个子块都不够，但它们合在一起其实已经覆盖了整个父块的核心信息。这时候与其分别处理每个子块，不如直接合并整个父块——这就是 AutoMerging 要做的事情。

## AutoMerging

### 失败场景
多个叶子块分散命中，单独送入 LLM 时信息仍割裂。

### 思路
先按叶子检索，再按“命中密度”决定是否提升到父块合并。

### 层级合并流程
1. 叶子检索
2. 统计各父块命中比例
3. 超过阈值则合并父块作为最终上下文

### 适用边界
- 适合：文档存在层级，且同一父块内证据分散
- 不适合：层级关系弱或阈值难以稳定设定

In [ ]:
# 核心逻辑：leaf 检索 + merge threshold
leaf_per_parent = {}
for c_idx, p_idx in child_to_parent.items():
    leaf_per_parent.setdefault(p_idx, []).append(c_idx)

def auto_merge_answer(question: str, merge_threshold: float = 0.5) -> str:
    hits = child_retriever.invoke(question)
    hit_set = {h.page_content for h in hits}
    hit_ids = [idx for idx, txt in enumerate(child_texts) if txt in hit_set]

    merged_parents = []
    for p_idx, leaves in leaf_per_parent.items():
        ratio = len([lid for lid in leaves if lid in hit_ids]) / max(1, len(leaves))
        if ratio >= merge_threshold:
            merged_parents.append(p_idx)

    if merged_parents:
        context = "\n\n".join(parent_texts[i] for i in merged_parents)
    else:
        context = "\n\n".join(child_texts[i] for i in hit_ids[:4])

    prompt = f"问题：{question}\n上下文：\n{context}\n请简洁作答。"
    return llm.invoke(prompt).content

rows = []
for question, expected in qna_dict.items():
    answer = auto_merge_answer(question, merge_threshold=0.5)
    rows.append(
        {
            "question": question,
            "llm_answer": answer,
            "expected_answer": expected,
            "rag_eval_results": simple_eval(answer, expected),
        }
    )

auto_merging_df = pd.DataFrame(rows)
auto_merging_df

### AutoMerging 结果分析

AutoMerging 的关键参数是 `merge_threshold`：阈值越低，越容易触发父块合并，上下文越完整但噪声也越多；阈值越高，越保守，可能回退到和 Small-to-Big 类似的行为。实际使用中需要根据文档结构和问题类型调试这个阈值。

到这里，三种检索时上下文增强方法都已经跑完了。它们的共同点是：**不改变检索算法本身，只在命中后恢复更多上下文**。但还有一种完全不同的思路：能不能在索引阶段就让每个 chunk 的 embedding "看到"完整文档？这就是接下来要介绍的 Late Chunking。

## 最终对比表

| 方法 | 典型修复问题 | 新增复杂度 | 最适合文档形态 |
|---|---|---|---|
| baseline | 无（作为对照） | 低 | 任意 |
| Sentence Window | 命中句缺邻域证据 | 低-中 | 连续叙述文本 |
| Small-to-Big | 小块准但不完整 | 中 | 段落层级清晰 |
| AutoMerging | 多叶子分散命中 | 中-高 | 树状层级明显 |
| Late Chunking（理论） | embedding 缺少文档全局信息 | 低（但模型受限） | 需长上下文 embedding 模型 |

## 前沿方法：Late Chunking（理论介绍）

### 传统流程的问题
传统 RAG 的流程是"先切分，再分别嵌入"——每个 chunk 独立通过 embedding 模型，丢失了跨 chunk 的语义关联。

### Late Chunking 思路
Late Chunking 颠倒了这个顺序：
1. 先将**整个文档**送入长上下文 embedding 模型（如 jina-embeddings-v2），获得每个 token 的上下文化表示
2. 再按预设边界切分 token embeddings，对每个 chunk 的 token embeddings 做池化得到 chunk embedding

这样每个 chunk 的向量都"见过"完整文档上下文，天然缓解了上下文割裂问题。

### 与本章方法的对比

| 维度 | Sentence Window / Small-to-Big / AutoMerging | Late Chunking |
|---|---|---|
| 增强时机 | 检索时（命中后恢复邻域） | 索引时（embedding 阶段） |
| 额外存储 | 需要维护邻居映射 / 父子关系 | 不需要 |
| 模型依赖 | 无特殊要求 | 需要长上下文 embedding 模型 |
| 实现复杂度 | 中 | 低（但模型选择受限） |

### 局限
- 依赖支持长上下文的 embedding 模型（如 jina-embeddings-v2、nomic-embed），OpenAI text-embedding-3 系列不直接支持此模式
- 文档超过模型上下文窗口时需要分段处理
- 目前 LangChain 生态无开箱即用的 Late Chunking 组件

### 本节为什么不做代码示例
Late Chunking 需要特殊的 embedding 模型和自定义 tokenizer 操作，与本教程统一使用 OpenAI embedding 的约定不兼容。此处仅作概念介绍，帮助读者建立"还有一类索引时上下文增强"的认知。

## 如何选择

- 如果问题经常“只差前后两句”：先用 Sentence Window。
- 如果文档天然有章节层级：优先 Small-to-Big。
- 如果证据常分散在同一父块多个子块：优先 AutoMerging。
- 若问题本质是多步推理或多轮交互，应转到流程增强或系统增强。
- 如果问题出在 chunk 本身缺少文档语境（脱离上下文后语义不完整），这属于**索引阶段**优化，应回到第 3 章的 CCH / Contextual Retrieval。本章的上下文增强解决的是"检索后恢复邻域"，而非"索引时缺少语境"。

## 学习检查点

- 你能区分“检索相关但上下文不足”与“流程不足”吗？
- 你能解释 Sentence Window 与 Small-to-Big 的关键差异吗？
- 你知道 AutoMerging 的阈值会如何影响召回上下文长度吗？

## 下一步

如果你发现问题的根源不是上下文不全，而是一次检索流程本身不够——比如需要多步推理、需要先评估检索质量再决定下一步——请继续学习 `2. 流程增强.ipynb`。